# SCRIPT DESCRIPTION

Initial script to set up VM in Workbench and start working with ASAP CRN datasets. This pipeline has been adapted from the ASAP CRN tutorials (https://github.com/ASAP-CRN/asap-crn-learning-lab?tab=readme-ov-file)

This script:

(1) describes how to create a python environment to analyse snRNAseq data

(2) create local copies of the ASAP CRN snRNAseq datasets

(3) extracts metadata and integrates it into the HVG adata

# 0. Environment info

The python environment needs to be created first if working for the first time on these scripts for scRNA seq analysis.
The environment.yml can be found in the same repo folder

### Steps to create a new environment

Open a terminal in Verily Workbench. From your Verily Workbench workspace app:
- Click the **+** to open a new window or navigate to Launcher window
- Launch **Terminal**

This opens a shell session within your workspace where you can run the commands below.

(1) — Create and activate the Conda environment
```bash 
    conda env create -f environment.yml 
    conda init
    #close and reopen a terminal
    conda activate my_sc_rnaseq_env
```

(2) —  Register the environment as a Jupyter kernel
```bash
python -m ipykernel install --user \
  --name sc_rnaseq \
  --display-name "Python (my_sc_rnaseq_env)"
```
After completing these steps, return to JupyterLab and select **Python (my_sc_rnaseq_env)** as the notebook kernel.

### If environment is already created and used before, launch JupyterLab and select **Python (my_sc_rnaseq_env)** as the notebook kernel.

 *NOTE: if any changes need to be made to the environment:
 
    - Create a new file with the edits (e.g. new dependencies)
    - Launch the Terminal
    ```bash
        conda env update -f environment.yml --prune
    ```


## 1. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import osc
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


## 2. Data structure

In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

#### 2.1 Define data from CRN Cloud that will be retrieved to make local copies

In [ ]:
## Build and set path to desired dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS" / "pmdbs-sc-rnaseq-v3"

workflow       = "pmdbs_sc_rnaseq"
dataset_team   = "cohort" #harmonised dataset containing all teams samples
dataset_source = "pmdbs"
dataset_type   = "sc-rnaseq"

bucket_name  = f"{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / bucket_name / workflow
print("Dataset Path:", dataset_path)

cohort_analysis_path = dataset_path / "cohort_analysis"
print("Contents of cohort_analysis:")
!ls {cohort_analysis_path}


#### 2.2. Define local folder for CRN Cloud files

In [ ]:
# Define a local path for workshop files
local_data_path = WS_FILES / "local_resources"

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    local_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {local_data_path}")

## 3. Copy Data Locally
Retrieve the curated input files that will be used for any analysis downstream:

- **`asap-{team}.final_metadata.csv`** → cell‑level metadata table
- **`asap-{team}.final.h5ad`** → AnnData object containing HVG expression data (top 3k genes) and associated annotations
- **`asap-{team}.merged-cleaned-unfiltered.h5ad`** → AnnData object containing the full gene expression matrix (before QC, pre-processing and integration/harmonisation)

These files are copied into the local `local_resources` directory.

In [ ]:
# Downloading obs field (cell metadata)
# Define the expected local path for the metadata file.
cell_metadata_local_path = local_data_path / f"asap-{dataset_team}.final_metadata.csv"\

# Check if the metadata file already exists locally.
if not cell_metadata_local_path.exists():
    # Construct the original path where the metadata file is stored.
    cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final_metadata.csv"

    # Use a shell command (`cp`) to copy the file from the original location
    # into the local workshop_files directory for analysis.
    !cp {cell_metadata_og_path} {cell_metadata_local_path}

In [ ]:
# Downloading the anndata object (hvg)
# Define the expected local path
adata_local_path = local_data_path / f"asap-{dataset_team}.final.h5ad"

# Check if the adata file already exists locally.
if not adata_local_path.exists():
    # Construct the original path where the metadata file is stored.
    adata_cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final.h5ad"

    # Use a shell command (`cp`) to copy the file from the original location
    # into the local workshop_files directory for analysis.
    !cp {adata_cell_metadata_og_path} {adata_local_path}


In [ ]:
# Downloading the anndata object (full gene matrix)
# Define the expected local path
full_adata_local_path = local_data_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"

# Check if the adata file already exists locally.
if not full_adata_local_path.exists():
    # Construct the original path where the metadata file is stored.
    full_adata_cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"

    # Use a shell command (`cp`) to copy the file from the original location
    # into the local workshop_files directory for analysis.
    !cp {full_adata_cell_metadata_og_path} {full_adata_local_path}


## 4. Metadata

Retrieve dataset metadata (sample and patient info) and save locally.

Section copied from ASAP CRN Tutorials

In [ ]:
#Define metadata folder path
ds_metadata_path = WS_ROOT / "release_resources/cohort-pmdbs-sc-rnaseq/metadata/release/v5.0.0"

#preview contents
!ls {ds_metadata_path} 

In [ ]:
# asap-crn

SAMPLE = pd.read_csv(ds_metadata_path / "SAMPLE.csv", low_memory=False)
SUBJECT = pd.read_csv(ds_metadata_path / "SUBJECT.csv", low_memory=False)
CLINPATH = pd.read_csv(ds_metadata_path / "CLINPATH.csv", low_memory=False)

source_tables = {
    "SAMPLE" : SAMPLE,
    "SUBJECT": SUBJECT,
    "CLINPATH": CLINPATH,
}


print("SAMPLE:", SAMPLE.shape)
print("SUBJECT:", SUBJECT.shape)
print("CLINPATH:", CLINPATH.shape)

In [ ]:
# define merge keys
subject_keys = [
    "ASAP_subject_id",
    "ASAP_dataset_id",
    "ASAP_team_id",
    "subject_id",
]

clinpath_keys = [
    "ASAP_team_id",
    "ASAP_dataset_id",
    "ASAP_subject_id",
    "subject_id",
    "source_subject_id",
]

In [ ]:
def collapse_duplicate_records(data, keys, table_name):
    """Reduce a table to one row per key, keeping the most complete record.

    Some subjects appear more than once with partially filled rows. For each
    group we keep the row with the most non-null fields, fill remaining gaps
    from its siblings, and record any field where the rows disagree.

    Returns (collapsed, conflicts, ties) -- always look at `conflicts`, since
    a non-empty result means two sources reported different values for the
    same subject.
    """
    work = data.copy()
    work = work.replace({None: pd.NA, "": pd.NA})

    metadata_columns = [c for c in work.columns if c not in keys]

    work["_fields_present"] = work[metadata_columns].notna().sum(axis=1)
    work["_source_row"] = np.arange(len(work))

    selected_records = []
    conflict_records = []
    tie_records = []

    group_key = keys[0] if len(keys) == 1 else list(keys)

    for key_values, group in work.groupby(group_key, dropna=False, sort=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)

        key_record = dict(zip(keys, key_values))
        max_fields = int(group["_fields_present"].max())

        top_rows = group.loc[group["_fields_present"].eq(max_fields)].sort_values("_source_row")
        selected = top_rows.iloc[0].copy()

        if len(top_rows) > 1:
            tie_records.append({
                **key_record,
                "table": table_name,
                "n_tied_rows": len(top_rows),
                "fields_present": max_fields,
                "source_rows": " | ".join(top_rows["_source_row"].astype(str).tolist()),
            })

        conflicting_fields = []

        for column in metadata_columns:
            values = group[column].dropna().drop_duplicates()

            if len(values) == 1:
                selected[column] = values.iloc[0]

            elif len(values) > 1:
                conflicting_fields.append(column)

                selected_top_values = top_rows[column].dropna()
                if len(selected_top_values):
                    selected[column] = selected_top_values.iloc[0]
                elif pd.isna(selected[column]):
                    selected[column] = values.iloc[0]

                conflict_records.append({
                    **key_record,
                    "table": table_name,
                    "field": column,
                    "n_unique_values": len(values),
                    "observed_values": " | ".join(repr(v) for v in values.tolist()),
                    "selected_value": repr(selected[column]),
                    "n_source_records": len(group),
                })
        selected_records.append(selected)

    collapsed = (
        pd.DataFrame(selected_records)
        .drop(columns=["_fields_present", "_source_row"])
        .reset_index(drop=True)
    )
    conflicts = pd.DataFrame(conflict_records)
    ties = pd.DataFrame(tie_records)

    if collapsed.duplicated(list(keys)).any():
        raise AssertionError(f"{table_name} cleaning did not produce one row per key.")

    return collapsed, conflicts, ties


In [ ]:
# ensure no duplicated subject_ids
SUBJECT_clean, SUBJECT_conflicts, SUBJECT_selection_ties = collapse_duplicate_records(
    SUBJECT,
    subject_keys,
    table_name="SUBJECT"
)

print("Original SUBJECT rows:", len(SUBJECT))
print("Clean SUBJECT rows:", len(SUBJECT_clean))
print("SUBJECT conflicts:", len(SUBJECT_conflicts))
print("SUBJECT top-record ties:", len(SUBJECT_selection_ties))

display(SUBJECT_conflicts)

expected_sample_rows = len(SAMPLE)

df = pd.merge(
    SAMPLE,
    SUBJECT_clean,
    on=subject_keys,
    how="left",
    validate="many_to_one",
)

assert len(df) == expected_sample_rows

In [ ]:
clinpath_duplicate_counts = (
    CLINPATH.groupby(clinpath_keys, dropna=False)
    .size()
    .reset_index(name="n_clinpath_rows")
    .query("n_clinpath_rows > 1")
    .sort_values("n_clinpath_rows", ascending=False)
)


CLINPATH_clean, CLINPATH_conflicts, CLINPATH_selection_ties = collapse_duplicate_records(
    CLINPATH,
    clinpath_keys,
    table_name="CLINPATH",
)

print("Original CLINPATH rows:", len(CLINPATH))
print("Clean CLINPATH rows:", len(CLINPATH_clean))
print("CLINPATH conflicts:", len(CLINPATH_conflicts))
print("CLINPATH top-record ties:", len(CLINPATH_selection_ties))

before_merge = len(df)

df = pd.merge(
    df,
    CLINPATH_clean,
    on=clinpath_keys,
    how="left",
    validate="many_to_one"
)

assert len(df) == before_merge

# create unique sample identifier
df["sample"] = df["ASAP_sample_id"] + "_" + df["replicate"]


#### 4.2. Harmonize Brain Region Labels

Map region_level_2 brain regions to a coaser taxonomy. 

In [ ]:
# Region fields arrive as "Name (ABBREV, UBERON:0000000)".
# Split them apart so we can filter on the plain name.

for i in [1, 2, 3]:
    col = f"region_level_{i}"
    df[[f"{col}_name", f"{col}_abbrev", f"{col}_ontology_term_id"]] = (
        df[col].str.extract(r"^(.*?) \((.*?), (.*?)\)$")
    )


In [ ]:
print(df.region_level_1_name.value_counts(), 
      df.region_level_2_name.value_counts(), 
      df.region_level_3_name.value_counts())

In [ ]:
df["region_level_2_name"] = df["region_level_2_name"].str.strip()

brain_simple = {
    "Prefrontal cortex":        "frontal_ctx",
    "Middle frontal gyrus":     "frontal_ctx",
    "Anterior cingulate gyrus": "cingulate_ctx",
    "Inferior parietal lobule": "parietal_ctx",
    "Middle temporal gyrus":    "temporal_ctx",
    "Hippocampus":              "subcortical",
    "Amygdala":                 "subcortical",
    "Putamen":                  "subcortical",
    "Substantia nigra":         "subcortical",
}

df["brain_region_simple"] = df["region_level_2_name"].map(brain_simple)

# Two different problems, so check them separately:
# a labeled region we forgot is a bug; a missing label is a data gap.
unknown = set(df["region_level_2_name"].dropna()) - set(brain_simple)
assert not unknown, f"Add to brain_simple: {unknown}"

n_missing = df["region_level_2_name"].isna().sum()
print(f"{n_missing} samples have no region label and will be excluded.")

In [ ]:
# df has one row per sample; adata.obs has 3.6M rows. These dicts are the bridge to annotate the cells in our adata.

assert df["sample"].is_unique, "sample IDs collapsed silently — check replicate suffix"

# Define sample to match
region_2_mapper_full = dict(zip(df["sample"], df["region_level_2_name"]))
region_2_mapper_simple = dict(zip(df["sample"], df["region_level_2_name"].map(brain_simple)))

# Parkinsons and control samples
condition_id_mapper = dict(zip(df["sample"], df["condition_id"]))
## case_id comes from gp2_phenotype, the harmonized GP2 label
case_id_mapper = dict(zip(df["sample"], df["gp2_phenotype"]))

# Detailed brain region mapper 
region_1_mapper = dict(zip(df["sample"], df["region_level_1_name"]))
region_3_mapper = dict(zip(df["sample"], df["region_level_3_name"]))

# Diagnoses
diagnoses_mapper = dict(zip(df["sample"], df["primary_diagnosis_x"]))

In [ ]:
#drop all empty columns
# Merging three wide tables leaves columns that are null for every sample.
df = df.convert_dtypes().dropna(axis=1, how="all")
df.head(5)

In [ ]:
# save metadata copy locally
dataset_metadata_file = local_data_path / "asap-cohort-dataset-metadata.csv"
df.to_csv(dataset_metadata_file)

## 5. Integrate HVG adata and metadata

There is a mismatch in the sample names between adata.obs and the sample metadata (ASAP_sample_ID + replicate). In adata.obs some samples are labelled as "PMDBS" and others as "PMBDS" (B and D have been transposed). Fix this before integrating the data by adding a new column in both df "sample_clean"

In [ ]:
# load the adata

adata = sc.read_h5ad(adata_local_path, backed="r")
adata

#### 5.1. Fix mismatched labelling by creating a new column with matching names

In [ ]:
# Extract unique samples from both sources
obs_samples = set(adata.obs["sample"].unique()) # adjust column name as needed
meta_samples = set(df["sample"].unique())

print(f"Total unique samples in adata.obs: {len(obs_samples)}")
print(f"Total unique samples in metadata df: {len(meta_samples)}")

# Check if metadata IDs are substrings inside any AnnData sample IDs
sample_meta_example = list(meta_samples)[0]
matching_in_obs = [s for s in obs_samples if str(sample_meta_example) in str(s)]

print(f"\nExample metadata ID: '{sample_meta_example}'")
print(f"Found as substring in adata.obs? {matching_in_obs[:3]}")

In [ ]:
# 1. Grab sorted lists of unique sample IDs from both objects
obs_list = sorted([str(s) for s in obs_samples])
meta_list = sorted([str(s) for s in meta_samples])

# Determine how many rows to inspect side-by-side
n_rows = min(len(obs_list), len(meta_list), 15)

# 2. Build a diagnostic comparison DataFrame
diag_df = pd.DataFrame({
    "obs_raw": obs_list[:n_rows],
    "meta_raw": meta_list[:n_rows],
    # Exact equality check
    "exact_match": [obs_list[i] == meta_list[i] for i in range(n_rows)],
    # Character length comparison
    "obs_len": [len(obs_list[i]) for i in range(n_rows)],
    "meta_len": [len(meta_list[i]) for i in range(n_rows)],
    # Repr showing hidden spaces, tabs, or newlines (e.g. 'ASAP_1 ' vs 'ASAP_1')
    "obs_repr": [repr(obs_list[i]) for i in range(n_rows)],
    "meta_repr": [repr(meta_list[i]) for i in range(n_rows)],
})

# Display all columns cleanly
pd.set_option('display.max_columns', None)
print(diag_df[["obs_raw", "meta_raw", "exact_match", "obs_len", "meta_len", "obs_repr", "meta_repr"]])

###-->>> there is a typo in most of the sample obs shows PMBDS while metadata is PMDBS (B and D are transposed)

In [ ]:
clean_obs = set(adata.obs["sample"].astype(str).str.strip().str.lower().unique())
clean_meta = set(df["sample"].astype(str).str.strip().str.lower().unique())

print(f"Raw set overlap count: {len(obs_samples.intersection(meta_samples))}")
print(f"Cleaned set overlap count: {len(clean_obs.intersection(clean_meta))}")

In [ ]:
# Function to count pattern occurrences in a series
def count_typo_patterns(series, name):
    pmbds_count = series.astype(str).str.contains("PMBDS").sum()
    pmdbs_count = series.astype(str).str.contains("PMDBS").sum()
    print(f"--- {name} ---")
    print(f"  Contains 'PMBDS': {pmbds_count}")
    print(f"  Contains 'PMDBS': {pmdbs_count}\n")

# 1. Count across UNIQUE sample IDs
obs_samples_series = pd.Series(list(obs_samples))
meta_samples_series = pd.Series(list(meta_samples))

print("=== UNIQUE SAMPLE ID COUNTS ===")
count_typo_patterns(obs_samples_series, "adata.obs Unique Samples")
count_typo_patterns(meta_samples_series, "metadata_df Unique Samples")

# 2. Count across ALL CELL ROWS in adata.obs
print("=== TOTAL CELL ROW COUNTS ===")
count_typo_patterns(adata.obs["sample"], "adata.obs (All Cells)")

#### The fix:

In [ ]:
# Fix the transposed typo in both objects
adata.obs["sample_clean"] = adata.obs["sample"].astype(str).str.replace("PMBDS", "PMDBS")
df["sample_clean"] = df["sample"].astype(str).str.replace("PMBDS", "PMDBS")

# Re-check overlap
clean_obs_set = set(adata.obs["sample_clean"].unique())
clean_meta_set = set(df["sample_clean"].unique())

print(f"Fixed overlap: {len(clean_obs_set.intersection(clean_meta_set))} / {len(clean_obs_set)} samples")


#### 5.2. Integrate HVG adata and metadata

To integrate the data we'll need to use the new columns created "sample_clean"

In [ ]:
# df has one row per sample; adata.obs has 3.6M rows. These dicts are the bridge to annotate the cells in our adata.

assert df["sample_clean"].is_unique, "sample IDs collapsed silently — check replicate suffix"

brain_simple = {
    "Prefrontal cortex":        "frontal_ctx",
    "Middle frontal gyrus":     "frontal_ctx",
    "Anterior cingulate gyrus": "cingulate_ctx",
    "Inferior parietal lobule": "parietal_ctx",
    "Middle temporal gyrus":    "temporal_ctx",
    "Hippocampus":              "subcortical",
    "Amygdala":                 "subcortical",
    "Putamen":                  "subcortical",
    "Substantia nigra":         "subcortical",
}

# Define sample to match
region_2_mapper_full = dict(zip(df["sample_clean"], df["region_level_2_name"]))
region_2_mapper_simple = dict(zip(df["sample_clean"], df["region_level_2_name"].map(brain_simple)))

# Parkinsons and control samples
condition_id_mapper = dict(zip(df["sample_clean"], df["condition_id"]))
## case_id comes from gp2_phenotype, the harmonized GP2 label
case_id_mapper = dict(zip(df["sample_clean"], df["gp2_phenotype"]))

# Detailed brain region mapper 
region_1_mapper = dict(zip(df["sample_clean"], df["region_level_1_name"]))
region_3_mapper = dict(zip(df["sample_clean"], df["region_level_3_name"]))

# Diagnoses
diagnoses_mapper = dict(zip(df["sample_clean"], df["primary_diagnosis_x"]))

# Biobank
biobank_mapper = dict(zip(df["sample_clean"], df["biobank_name"]))

In [ ]:
# Map samples to metadata
adata.obs["region_level_2"] = adata.obs["sample_clean"].map(region_2_mapper_full)
adata.obs["brain_region_simple"] = adata.obs["sample_clean"].map(region_2_mapper_simple)
adata.obs["case_id"] = adata.obs["sample_clean"].map(case_id_mapper)
adata.obs["condition_id"] = adata.obs["sample_clean"].map(condition_id_mapper)
adata.obs["region_level_1"] = adata.obs["sample_clean"].map(region_1_mapper)
adata.obs["region_level_3"] = adata.obs["sample_clean"].map(region_3_mapper)
adata.obs["primary_diagnosis"] = adata.obs["sample_clean"].map(diagnoses_mapper)
adata.obs["biobank_name"] = adata.obs["sample_clean"].map(biobank_mapper)


In [ ]:
""" 
Save adata with dataset metadata already integrated for future usage.

# Save this intermediate anndata which will contain:
# - All brain regions
# - Highly variable genes (HVGs) only
# - Normalized and log-transformed expression values
# - PCA and other preprocessing results stored in .obsm
"""
anndata_filename = local_data_path / f"asap-cohort.final.metadataset-integrated.h5ad"
updated_metadata_file = local_data_path / f""
adata.write_h5ad(anndata_filename)

metadata_filename = local_data_path / f"asap-cohort-dataset-metadata_merger_file.csv"


In [ ]:
adata.file.close()  # close the original adata file